<a href="https://colab.research.google.com/github/internshipinabook/nlp-internship-in-a-book/blob/main/notebooks/week1_text_is_not_data_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Colab setup (skip if running locally) ──────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/nlp-internship-in-a-book.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 1 — Text Is Not Data (STARTER)
### The NLP Internship | LinguaAI Labs

This week: tokenisation, vocabulary statistics, Zipf's Law, OOV rates by language, and your first vocabulary explorer.

**Fill in every `# YOUR CODE HERE` block. Run cells in order.**

In [ ]:
# ── SETUP ─────────────────────────────────────────────────────────────────────
import sys, subprocess, importlib
for pkg, imp in [("spacy","spacy"),("nltk","nltk"),("langdetect","langdetect")]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
import spacy, nltk, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
import warnings; warnings.filterwarnings("ignore")
for r in ["punkt","punkt_tab","stopwords","wordnet"]:
    nltk.download(r, quiet=True)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")
np.random.seed(42)
print("Setup complete ✅")

## Part 1 — Load the Corpus

In [ ]:
import pandas as pd, os

# Smart loader: local package first → GitHub → sample data fallback
LOCAL_PATH = '../data/support_tickets.csv'
REMOTE_URL = 'https://raw.githubusercontent.com/internshipinabook/nlp-internship-in-a-book/main/data/support_tickets.csv'

df = None
if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local package ✅')
else:
    try:
        df = pd.read_csv(REMOTE_URL)
        print(f'Loaded from GitHub repository ✅')
    except Exception as e:
        print(f'⚠️  Dataset not yet online ({e})')
        print('Generating sample data — upload support_tickets.csv to data/ folder')
        import numpy as np; np.random.seed(42)
        tickets = [
            'My account login is not working, please help',
            'I was charged twice for my subscription this month',
            'The app crashes whenever I try to upload a file',
            'How do I cancel my subscription?',
            'Great service, very happy with the product!',
            'I cannot reset my password, the email never arrives',
            'The dashboard is loading very slowly today',
            'I need a refund for my last order',
            'Feature request: can you add dark mode?',
            'Error 500 when trying to export my data',
        ] * 50
        np.random.shuffle(tickets)
        df = pd.DataFrame({
            'ticket_id':  range(1, 501),
            'text':       tickets[:500],
            'category':   np.random.choice(['billing','technical','account','general'], 500),
            'sentiment':  np.random.choice(['positive','negative','neutral'], 500, p=[0.3,0.4,0.3]),
            'priority':   np.random.choice(['low','medium','high'], 500),
        })
        print('Sample dataset ready (500 tickets) ✅')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()


## Part 2 — Three Tokenisation Approaches

> ⏸️ **Pause and Predict**
> Before running: what problem do you expect when splitting on spaces (naive tokenisation)? Write your prediction in the cell below.

In [ ]:
# YOUR PREDICTION (markdown or comment):
# ...

ticket = "Customer can't access her account. Says it's been 3 days. Very frustrated!!!"

# Approach 1: naive split on spaces
naive_tokens = None  # YOUR CODE HERE — hint: ticket.split()

# Approach 2: spaCy tokenisation
doc = nlp(ticket)
spacy_tokens = None  # YOUR CODE HERE (list of token.text, no punctuation)

# Approach 3: spaCy without punctuation
clean_tokens = None  # YOUR CODE HERE

print(f"Naive  ({len(naive_tokens)} tokens): {naive_tokens[:6]}")
print(f"spaCy  ({len(spacy_tokens)} tokens): {spacy_tokens[:6]}")
print(f"Clean  ({len(clean_tokens)} tokens): {clean_tokens[:6]}")

## Part 3 — Corpus Vocabulary

> 🧠 **What Will This Output?**
> Before running the full corpus tokenisation, predict: will the vocabulary size be larger or smaller than 10,000 unique tokens? Why?

In [ ]:
# Tokenise all 50,000 tickets using nlp.pipe (fast batch processing)
# This takes 2-3 minutes. Read the note below while it runs.
print("Tokenising corpus — this takes 2-3 minutes...")

all_tokens = []
for doc in nlp.pipe(df["text"].fillna(""), batch_size=500):
    tokens = None  # YOUR CODE HERE (lowercase, no punct, len > 1)
    all_tokens.extend(tokens)

word_freq = Counter(all_tokens)
print(f"\nTotal tokens:  {len(all_tokens):,}")
print(f"Unique types:  {len(word_freq):,}")
print(f"Type-Token Ratio: {len(word_freq)/len(all_tokens):.4f}")
print(f"\nTop 20 words:")
for word, count in word_freq.most_common(20):
    print(f"  {word:<20} {count:,}")

## Part 4 — Zipf's Law

In [ ]:
# Plot the Zipf distribution
ranks = range(1, 5001)
freqs = [word_freq.most_common(5000)[i][1] for i in range(5000)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
axes[0].plot(list(ranks), freqs, color="#2E75B6", linewidth=1)
axes[0].set_title("Word Frequency by Rank", fontweight="bold", loc="left")
axes[0].set_xlabel("Rank"); axes[0].set_ylabel("Frequency")
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)

# Log-log scale — reveals the power law
axes[1].loglog(list(ranks), freqs, color="#2E75B6", linewidth=1)
axes[1].set_title("Zipf Distribution (Log-Log)", fontweight="bold", loc="left")
axes[1].set_xlabel("Rank (log)"); axes[1].set_ylabel("Frequency (log)")
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("outputs/zipf_distribution.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
# Write a markdown observation: what does the log-log straight line tell you about word frequency?

## Part 5 — OOV Rates by Language

> ⏸️ **Pause and Predict**
> Which language group do you expect to have the highest OOV rate with the English spaCy model — and why?

In [ ]:
# Measure out-of-vocabulary rate per language group
print("OOV rates by language (sample of 200 per group):\n")

for language, group in df.groupby("language_detected"):
    sample = group["text"].sample(min(200, len(group)), random_state=42).tolist()
    oov_rates = []
    for doc in nlp.pipe(sample, batch_size=50):
        tokens = [t for t in doc if not t.is_punct and len(t.text) > 1]
        if tokens:
            oov_rate = None  # YOUR CODE HERE — fraction where t.is_oov is True
            oov_rates.append(oov_rate)
    mean_oov = np.mean(oov_rates) if oov_rates else 0
    print(f"  {language:<12}: {mean_oov:.1%} OOV ({len(group):,} tickets)")

## Part 6 — Vocabulary Explorer

In [ ]:
def find_neighbours(word, token_list, window=5, top_n=10):
    """
    Return the top_n words that appear within `window` positions of `word`.
    """
    # YOUR CODE HERE
    # Hint: iterate through token_list; when you find the word,
    # collect context tokens in positions [i-window : i] and [i+1 : i+window+1]
    neighbours = Counter()
    word_lower = word.lower()
    # ...
    return neighbours.most_common(top_n)

# Test your vocabulary explorer
for query in ["account", "payment", "abeg"]:
    print(f"\nNeighbours of '{query}':")
    for word, count in find_neighbours(query, all_tokens):
        print(f"  {word:<20} ({count:,}x)")

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Treating raw text as ready to use | Raw text has encoding issues, HTML tags, emojis, and inconsistent whitespace. It is never ready as-is. | Always inspect raw text character by character before processing |
| Using word count as a proxy for information | A 500-word ticket can say nothing. A 10-word ticket can contain the entire problem. Length is not content. | Extract meaning, not metrics |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Load the dataset. Write a one-paragraph memo describing what you find — not statistics, but what the data is actually about and what problem it could help solve.

> 🔬 **Advanced Path**
> Find the five most common words in the dataset. Now remove stop words and find the five most meaningful words. What changed, and why does it matter?

## ✅ What You Can Do After This Week

- Tokenise a corpus using spaCy and measure vocabulary statistics
- Explain Zipf's Law and what it implies about word frequency
- Identify OOV rates by language and explain their implications
- Build a vocabulary explorer using co-occurrence counts

---
## ✅ Week 1 Complete
**Next:** `week2_cleaning_text_STARTER.ipynb`

---
*Add `week1_text_is_not_data_STARTER.ipynb` to your internship portfolio.*

*The NLP Internship · LinguaAI Labs · github.com/InternshipInABook*

## ✅ By completing Week 1, you can now:

- Explain why raw text cannot be fed directly into a machine learning model
- Identify at least three data quality issues in a raw corpus
- Apply tokenisation and explain the difference between word and subword tokens
- Write a data quality memo that a product manager could act on

📤 **GitHub:** Push week1_text_is_not_data_STARTER.ipynb. Commit: "Week 1: Corpus described, first tokens inspected"


---

## 📚 Get the Full Book

This notebook is part of **The NLP Internship** — Book 2 of the InternshipInABook™ Series.

The full book includes:
- Complete week-by-week narrative and explanations
- All STOP AND TRACE code walkthroughs
- Fairness briefs, model cards, and deployment guides
- Certificate of Completion

👉 **[Get the book on Selar →](https://selar.com/8440iqfr61)**

---
*InternshipInABook™ Series · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook)*
